In [ ]:
# KAGGLE CELL 1
import os

# 1. Install required dependencies
!pip install -q accelerate datasets soundfile librosa pandas wandb

# 2. Clone F5-TTS (Flow-Matching Architecture)
!git clone https://github.com/SWivid/F5-TTS.git
%cd F5-TTS
!pip install -e .

# 3. Download and extract your exact dataset
!gdown fsdfsdf

import zipfile
os.makedirs('./dataset', exist_ok=True)
with zipfile.ZipFile('dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('./dataset')

In [ ]:
# KAGGLE/COLAB CELL 2
import os
import json
import librosa
import soundfile as sf
import pandas as pd
from datasets import Dataset, Audio
from tqdm.notebook import tqdm

print("🎧 Building F5-TTS hardcoded dataset structure...")

root_dir = "./dataset"
base_dir = "./data/mongolian_char"
wav_dir = os.path.join(base_dir, "wavs")

os.makedirs(wav_dir, exist_ok=True)

records =[]
vocab = set()
durations_dict = {} # ⬅️ THE MISSING DICTIONARY

# 1. Parse metadata & process audio
for dirpath, _, filenames in os.walk(root_dir):
    for filename in filenames:
        if filename.endswith("metadata.txt"):
            txt_path = os.path.join(dirpath, filename)
            with open(txt_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line: continue
                    parts = line.split("|")
                    if len(parts) >= 2:
                        wav_filename, text = parts[0], parts[1]
                        full_wav_path = os.path.join(dirpath, wav_filename)

                        if os.path.exists(full_wav_path):
                            audio, sr = librosa.load(full_wav_path, sr=24000, mono=True)
                            audio = librosa.util.normalize(audio)

                            duration = librosa.get_duration(y=audio, sr=24000)

                            out_wav_path = os.path.join(wav_dir, f"{len(records):05d}.wav")
                            sf.write(out_wav_path, audio, 24000, subtype='PCM_16')

                            records.append({
                                "audio": out_wav_path,
                                "text": text,
                                "duration": duration
                            })
                            # ⬅️ Map the path to duration for F5-TTS batching
                            durations_dict[out_wav_path] = duration
                            vocab.update(list(text))

# 2. Generate vocab.txt
vocab_list = sorted(list(vocab))
vocab_path = os.path.join(base_dir, "vocab.txt")
with open(vocab_path, "w", encoding="utf-8") as f:
    for c in vocab_list:
        f.write(c + "\n")

# 3. Generate duration.json (What just crashed your script)
duration_path = os.path.join(base_dir, "duration.json")
with open(duration_path, "w", encoding="utf-8") as f:
    json.dump(durations_dict, f, indent=4)
print(f"✅ Saved durations to {duration_path}")

# 4. Create Hugging Face Dataset format
print("📦 Building Arrow Dataset...")
df = pd.DataFrame(records)
hf_dataset = Dataset.from_pandas(df)
hf_dataset = hf_dataset.cast_column("audio", Audio(sampling_rate=24000))

raw_dir = os.path.join(base_dir, "raw")
hf_dataset.save_to_disk(raw_dir)

print(f"✅ Data fully saved to F5-TTS's required path: {base_dir}")

🎧 Building F5-TTS hardcoded dataset structure...
✅ Saved durations to ./data/mongolian_char/duration.json
📦 Building Arrow Dataset...


Saving the dataset (0/1 shards):   0%|          | 0/232 [00:00<?, ? examples/s]

✅ Data fully saved to F5-TTS's required path: ./data/mongolian_char


In [ ]:
# KAGGLE/COLAB CELL 3
import yaml
import glob
import os
import json
from datasets import Dataset, Audio

# ---------------------------------------------------------
# 1. UPDATE F5-TTS CONFIGURATION
# ---------------------------------------------------------
config_dir = "src/f5_tts/configs"
base_configs = glob.glob(f"{config_dir}/*Base.yaml")
base_config_path = base_configs[0]

with open(base_config_path, "r") as f:
    config = yaml.safe_load(f)

# --- INJECT DATASET CONFIG ---
config["datasets"]["name"] = "mongolian"

# --- CONFIGURE CHARACTER TOKENIZER ---
if "model" not in config: config["model"] = {}
config["model"]["tokenizer"] = "char"

# --- 🚀 UPDATED TRAINING HYPERPARAMETERS ---
config["datasets"]["batch_size_per_gpu"] = 3840
config["datasets"]["batch_size_type"] = "frame"
config["optim"]["epochs"] = 150
config["optim"]["learning_rate"] = 5e-5

# --- 💾 CHECKPOINT SAVING SETTINGS ---
if "ckpts" not in config: config["ckpts"] = {}
config["ckpts"]["last_per_updates"] = 1200
config["ckpts"]["save_per_updates"] = 1500
config["ckpts"]["keep_last_n_checkpoints"] = 3

custom_config_name = "F5TTS_Mongolian.yaml"
with open(os.path.join(config_dir, custom_config_name), "w") as f:
    yaml.dump(config, f)

print(f"✅ Custom config {custom_config_name} generated.")

# ---------------------------------------------------------
# 2. FIX DATASET COLUMNS AND PATHS
# ---------------------------------------------------------
base_dir = "/content/F5-TTS/data/mongolian_char"
raw_dir = os.path.join(base_dir, "raw")
wav_dir = os.path.join(base_dir, "wavs")

print("🛠️ Fixing dataset structure for F5-TTS...")
ds = Dataset.load_from_disk(raw_dir)

# Safely extract audio paths depending on what state the dataset is currently in
correct_audio_paths =[]
if "audio_path" in ds.column_names:
    # If already fixed previously, just ensure the paths are absolute
    for p in ds["audio_path"]:
        correct_audio_paths.append(os.path.join(wav_dir, os.path.basename(p)))
elif "audio" in ds.column_names:
    # If raw, extract the path directly from the Hugging Face Audio dict
    ds = ds.cast_column("audio", Audio(decode=False))
    for i, row in enumerate(ds["audio"]):
        if row is not None and "path" in row and row["path"]:
            correct_audio_paths.append(os.path.join(wav_dir, os.path.basename(row["path"])))
        else:
            # Fallback in case Hugging Face stripped the path
            correct_audio_paths.append(os.path.join(wav_dir, f"{i:05d}.wav"))
else:
    raise ValueError("Dataset has neither 'audio' nor 'audio_path' columns!")

texts = ds["text"]
durations = [float(d) for d in ds["duration"]]

# Rebuild the dataset explicitly for F5-TTS
fixed_ds = Dataset.from_dict({
    "audio_path": correct_audio_paths,
    "text": texts,
    "duration": durations
})

# Overwrite on disk
fixed_ds.save_to_disk(raw_dir)

# ---------------------------------------------------------
# 3. FIX DURATION.JSON
# ---------------------------------------------------------
# F5-TTS expects a strict dictionary format {"duration": [float, float, ...]}
duration_path = os.path.join(base_dir, "duration.json")
with open(duration_path, "w", encoding="utf-8") as f:
    json.dump({"duration": durations}, f)

print(f"✅ Fixed! Dataset ready and duration.json has {len(durations)} valid float entries.")

✅ Custom config F5TTS_Mongolian.yaml generated.
🛠️ Fixing dataset structure for F5-TTS...


Saving the dataset (0/1 shards):   0%|          | 0/232 [00:00<?, ? examples/s]

✅ Fixed! Dataset ready and duration.json has 232 valid float entries.


In [ ]:
# !rm -rf /content/F5-TTS/ckpts/F5TTS_v1_Base_vocos_char_mongolian

In [ ]:
# KAGGLE/COLAB CELL 4
!WANDB_DISABLED=true accelerate launch \
    --num_processes=1 \
    --num_machines=1 \
    --mixed_precision=fp16 \
    --dynamo_backend=no \
    src/f5_tts/train/train.py \
    --config-name F5TTS_Mongolian.yaml

In [ ]:
# KAGGLE CELL 5
import os

# 1. Corrected checkpoint directory
checkpoint_dir = "./ckpts/F5TTS_v1_Base_vocos_char_mongolian/"
checkpoints = sorted([f for f in os.listdir(checkpoint_dir) if f.endswith(".pt")])
latest_ckpt = os.path.join(checkpoint_dir, checkpoints[-1])

print(f"🔊 Running inference on: {latest_ckpt}")

# 2. Corrected data paths (mongolian_char instead of mongolian_custom)
ref_audio = "./data/mongolian_char/wavs/00000.wav"
ref_text = "Үүнийг өөрийн аудионд тохируулан солино уу." # Replace with actual text of the ref_audio
gen_text = "Өнөөдөр гадаа хасах хорин хэм хүйтэн байна."

# Execute standard inference
# 3. Changed --model to "F5TTS_Mongolian" so it finds your custom .yaml config
!python src/f5_tts/infer/infer_cli.py \
    --model "F5TTS_Mongolian" \
    --ckpt_file "{latest_ckpt}" \
    --vocab_file "./data/mongolian_char/vocab.txt" \
    --ref_audio "{ref_audio}" \
    --ref_text "{ref_text}" \
    --gen_text "{gen_text}" \
    --output_dir "output"

import IPython.display as ipd

# 4. Corrected the filename for the audio player
ipd.display(ipd.Audio("output/infer_cli_basic.wav", rate=24000))

🔊 Running inference on: ./ckpts/F5TTS_v1_Base_vocos_char_mongolian/model_last.pt
Download Vocos from huggingface charactr/vocos-mel-24khz
Using F5TTS_Mongolian...

vocab :  ./data/mongolian_char/vocab.txt
token :  custom
model :  ./ckpts/F5TTS_v1_Base_vocos_char_mongolian/model_last.pt 

Voice: main
ref_audio  ./data/mongolian_char/wavs/00000.wav
Converting audio...
Using custom reference text...

ref_text   Үүнийг өөрийн аудионд тохируулан солино уу. 
ref_audio_ /tmp/tmpk2lizo6n.wav 


No voice tag found, using main.
Voice: main
gen_text 0 Өнөөдөр гадаа хасах хорин хэм хүйтэн байна.


Generating audio in 1 batches...
100% 1/1 [00:07<00:00,  7.26s/it]
output/infer_cli_basic.wav


In [ ]:
!ls -R /content/

# Continue Training

In [ ]:
# KAGGLE/COLAB CELL 4 (RESUME TRAINING)
import os

checkpoint_dir = "./ckpts/F5TTS_v1_Base_vocos_char_mongolian"
last_ckpt_path = os.path.join(checkpoint_dir, "model_last.pt")

if os.path.exists(last_ckpt_path):
    print(f"✅ FOUND CHECKPOINT: {last_ckpt_path}")
    print("🚀 F5-TTS will automatically load this and resume your training!")
else:
    print("⚠️ No model_last.pt found! Training will start from step 0 (Pre-trained Base).")

# Launch the exact same training command. F5-TTS handles the resuming automatically.
!WANDB_DISABLED=true accelerate launch \
    --num_processes=1 \
    --num_machines=1 \
    --mixed_precision=fp16 \
    --dynamo_backend=no \
    src/f5_tts/train/train.py \
    --config-name F5TTS_Mongolian.yaml

Epoch 36/150:  82% 55/67 [00:40<00:07,  1.51update/s, loss=1.46, update=2400]